# Notebook 01 — Data Download & Storage

**Goal**: Download 5 years of daily OHLCV data for 120+ NSE stocks + external indices.

---

## What We Download

| Source | Tickers | Purpose |
|--------|---------|---------|
| NSE Equities | 120+ stocks (.NS suffix) | Main prediction targets |
| ^NSEI | Nifty 50 index | Benchmark & cross-asset feature |
| ^NSEBANK | Bank Nifty index | Sector benchmark |
| ^INDIAVIX | India VIX | Volatility regime |
| GC=F | Gold futures | Safe-haven indicator |
| CL=F | Crude oil | Macro factor |
| USDINR=X | USD/INR | Currency impact |

## Data Source

We use **yfinance** which pulls data from Yahoo Finance. For NSE stocks,
append `.NS` to the ticker symbol (e.g., `RELIANCE.NS`).

## Storage Format

All data is saved as **Parquet** files (columnar, compressed, fast I/O).
- `data/raw/` — raw OHLCV as downloaded
- `data/external/` — benchmark indices and macro data

## Important Notes
- Download takes ~10-15 minutes for 120+ stocks
- Rate limiting is built in (0.5s between batches)
- Failed downloads are logged and retried
- Re-running this notebook will skip already-downloaded files (set `force=True` to re-download)

In [4]:
import os, sys, gc, time
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import yaml
from pathlib import Path

from src.data import TickerUniverse, DataDownloader
from src.utils.constants import *

# Stabilize kernel after heavy imports
gc.collect()
time.sleep(0.1)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/anto/Trading_Project/masters_trading_ai


## Step 1 — Load Ticker Universe

The ticker universe is defined in `config/tickers.yaml` with 5 buckets.
Each bucket targets different market segments for diversification.

In [7]:
universe = TickerUniverse()
print(repr(universe))

# Show tickers by bucket
for bucket in universe.buckets:
    tickers = universe.get_tickers(bucket)
    print(f"\n{bucket.upper()} ({len(tickers)} tickers):")
    print(f"  {', '.join(tickers[:10])}{'...' if len(tickers) > 10 else ''}")

TickerUniverse(124 tickers: {'large_cap': 30, 'banking': 23, 'mid_cap': 27, 'high_volatility': 21, 'commodities': 23})

LARGE_CAP (30 tickers):
  RELIANCE.NS, TCS.NS, HDFCBANK.NS, INFY.NS, ICICIBANK.NS, HINDUNILVR.NS, ITC.NS, BHARTIARTL.NS, SBIN.NS, BAJFINANCE.NS...

BANKING (23 tickers):
  HDFCLIFE.NS, SBILIFE.NS, BANKBARODA.NS, CANBK.NS, PNB.NS, ICICIGI.NS, CHOLAFIN.NS, SHRIRAMFIN.NS, JIOFIN.NS, LICI.NS...

MID_CAP (27 tickers):
  NAUKRI.NS, INDHOTEL.NS, TRENT.NS, PIDILITIND.NS, HAVELLS.NS, GODREJCP.NS, DMART.NS, DLF.NS, LODHA.NS, TVSMOTOR.NS...

HIGH_VOLATILITY (21 tickers):
  ADANIENT.NS, ADANIPORTS.NS, ADANIGREEN.NS, ADANIPOWER.NS, TMCV.NS, TMPV.NS, ETERNAL.NS, PAYTM.NS, INDIAMART.NS, MAZDOCK.NS...

COMMODITIES (23 tickers):
  TATASTEEL.NS, JSWSTEEL.NS, HINDALCO.NS, VEDL.NS, HINDZINC.NS, COALINDIA.NS, ONGC.NS, BPCL.NS, IOC.NS, GAIL.NS...


## Step 2 — Download Equity Data

The `DataDownloader` class handles:
- Batch downloading with configurable batch size
- Rate limiting to avoid API throttling
- Automatic retry with exponential backoff
- Progress tracking

Data is saved to `data/raw/{TICKER}.parquet`

In [3]:
downloader = DataDownloader(universe)

# Download all equity + external tickers
print(f"Downloading {len(universe)} equity tickers + {len(universe.external)} external tickers...")
print(f"Period: 10 years (2016-2026)")
print(f"This may take 15-20 minutes.\n")

results = downloader.download_all(years=10)

# Summary
print(f"\nSuccess: {len(results['success'])}, Failed: {len(results['failed'])}")
if results['failed']:
    print(f"Failed tickers: {results['failed']}")

results['summary']

Period: 10 years (2016-2026)
This may take 15-20 minutes.

Period: 2016-02-15 to 2026-02-12
------------------------------------------------------------


Trading tickers: 100%|██████████| 124/124 [01:38<00:00,  1.26it/s]


External tickers: 100%|██████████| 6/6 [00:04<00:00,  1.45it/s]


Download complete: 130 success, 0 failed

Success: 130, Failed: 0


,Ticker,Bucket,Rows,Start,End,Status
0,RELIANCE.NS,large_cap,2469,2016-02-15,2026-02-11,✓
1,TCS.NS,large_cap,2469,2016-02-15,2026-02-11,✓
2,HDFCBANK.NS,large_cap,2469,2016-02-15,2026-02-11,✓
3,INFY.NS,large_cap,2469,2016-02-15,2026-02-11,✓
4,ICICIBANK.NS,large_cap,2469,2016-02-15,2026-02-11,✓
...,...,...,...,...,...,...
119,BOSCHLTD.NS,commodities,2469,2016-02-15,2026-02-11,✓
120,DIVISLAB.NS,commodities,2469,2016-02-15,2026-02-11,✓
121,ZYDUSLIFE.NS,commodities,2469,2016-02-15,2026-02-11,✓
122,BAJAJ-AUTO.NS,commodities,2469,2016-02-15,2026-02-11,✓


## Step 3 — Download External / Benchmark Data

These are cross-asset features used by the model:
- **Nifty 50** (`^NSEI`): Market benchmark for alpha/beta calculation
- **Bank Nifty** (`^NSEBANK`): Banking sector benchmark
- **India VIX** (`^INDIAVIX`): Fear gauge — high VIX = high fear
- **Gold** (`GC=F`): Safe-haven proxy
- **Crude Oil** (`CL=F`): Energy/inflation proxy
- **USD/INR** (`USDINR=X`): Currency impact on FII flows

In [5]:
# External tickers were already downloaded by download_all()
# Load and inspect them
print("External benchmark / cross-asset data:")
for ticker in universe.external:
    df = downloader.load_ticker(ticker)
    if df is not None:
        print(f"  {ticker:15s}: {len(df)} trading days, "
              f"{df.index[0].date()} → {df.index[-1].date()}")
    else:
        print(f"  {ticker:15s}: ❌ NOT FOUND")

External benchmark / cross-asset data:
  ^NSEI          : 2463 trading days, 2016-02-15 → 2026-02-11
  ^NSEBANK       : 2260 trading days, 2016-02-19 → 2026-02-11
  ^INDIAVIX      : 2449 trading days, 2016-02-15 → 2026-02-11
  USDINR=X       : 2600 trading days, 2016-02-15 → 2026-02-11
  GC=F           : 2512 trading days, 2016-02-16 → 2026-02-11
  CL=F           : 2513 trading days, 2016-02-16 → 2026-02-11


## Step 4 — Quick Data Validation

Let's load a few tickers and check data quality before moving to the EDA notebook.

In [6]:
# Load a sample ticker
sample_tickers = ["RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "ICICIBANK.NS"]

for ticker in sample_tickers:
    df = downloader.load_ticker(ticker)
    if df is not None:
        missing_pct = df.isnull().sum().sum() / (len(df) * len(df.columns)) * 100
        print(f"{ticker:15s} | {len(df):5d} days | "
              f"{df.index[0].date()} → {df.index[-1].date()} | "
              f"Missing: {missing_pct:.2f}%")
    else:
        print(f"{ticker:15s} | ❌ NOT FOUND")

RELIANCE.NS     |  2469 days | 2016-02-15 → 2026-02-11 | Missing: 0.00%
TCS.NS          |  2469 days | 2016-02-15 → 2026-02-11 | Missing: 0.00%
HDFCBANK.NS     |  2469 days | 2016-02-15 → 2026-02-11 | Missing: 0.00%
INFY.NS         |  2469 days | 2016-02-15 → 2026-02-11 | Missing: 0.00%
ICICIBANK.NS    |  2469 days | 2016-02-15 → 2026-02-11 | Missing: 0.00%


## Step 5 — Data Files on Disk

Let's verify the Parquet files were saved correctly.

In [8]:
raw_files = list((DATA_DIR / "raw").glob("*.parquet"))
ext_files = list((DATA_DIR / "external").glob("*.parquet"))

total_size_mb = sum(f.stat().st_size for f in raw_files + ext_files) / 1e6

print(f"Raw equity files:     {len(raw_files)}")
print(f"External data files:  {len(ext_files)}")
print(f"Total disk usage:     {total_size_mb:.1f} MB")

print(f"\nExternal files:")
for f in sorted(ext_files):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

Raw equity files:     124
External data files:  6
Total disk usage:     14.3 MB

External files:
  CL_F.parquet (117.1 KB)
  GC_F.parquet (111.1 KB)
  IDX_INDIAVIX.parquet (76.0 KB)
  IDX_NSEBANK.parquet (110.6 KB)
  IDX_NSEI.parquet (123.3 KB)
  USDINR_X.parquet (110.8 KB)


## ✅ Data Download Complete!

**What we did:**
1. Downloaded OHLCV data for 120+ NSE stocks (5 years)
2. Downloaded 6 external benchmark/macro series
3. Saved everything as Parquet files in `data/raw/` and `data/external/`
4. Validated data quality (date ranges, missing values)

**Next:** Proceed to **Notebook 02 — EDA & Data Cleaning** to explore the data and handle quality issues.

---
*Data sourced from Yahoo Finance via yfinance. For educational purposes only.*